### Data Fetching

In [ ]:
import psycopg2
import pandas as pd

In [ ]:
def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

In [ ]:
keywords = ["OTN"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df

In [ ]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

In [ ]:
import pandas as pd
import json

df_otn = df.copy(deep=True)

valid_mask = df_otn['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_otn[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename']
    }

    cctv_data = row['json_data'].get('otn', {})

    for key, value in cctv_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_otn = pd.DataFrame(rows)

df_otn = df_otn.drop(
    columns=["pm_order_no", "reference_document"],
    errors="ignore"
)

df_otn

In [ ]:
df_otn.columns

In [ ]:
def flatten_procedures(proc_data):
    """
    Flattens JSON into 'module.step.status' and 'module.step.remarks' format.
    """
    flat_data = {}
    
    if isinstance(proc_data, str):
        try:
            proc_data = json.loads(proc_data)
        except:
            return {}
            
    if not isinstance(proc_data, dict): 
        return flat_data

    def walk(d, parent_key=''):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            
            if isinstance(v, dict):
                if 'status' in v:
                    flat_data[f"{new_key}.status"] = v.get('status')
                    flat_data[f"{new_key}.remarks"] = v.get('remarks')
                else:
                    walk(v, new_key)
            else:
                flat_data[new_key] = v

    walk(proc_data)
    return flat_data

def parse_json(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return {}
    return x

df_otn["procedures"] = df_otn["procedures"].apply(parse_json)

df_proc_flat = pd.DataFrame(
    df_otn["procedures"].apply(flatten_procedures).tolist(), 
    index=df_otn.index
)

df_final = pd.concat([df_otn.drop(columns=['procedures']), df_proc_flat], axis=1)

print(df_final.columns)

In [ ]:
list(df_final.columns)

In [ ]:
output_file = f"../../output/snc/otn.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='otn', index=False)

print(f"Saved excel to {output_file}")